# DRBL Stage 1 - Model Testing

This Jupyter Notebook is converted from the original shell script [test.sh](file:///d:/PHD/Recreations/DRBN/CVPR-2020-Semi-Low-Light/DRBL-stage1/src/test.sh). It is used to test the Deep Recursive Band Network (DRBN) Stage 1 model.

### Command executed in the original shell script:
```bash
CUDA_VISIBLE_DEVICES=5 python main_test.py --save_results --test_only --save test_1117_ms_re_v8_test --data_range 1-800/1-200 --save Full_results
```

> **Note**: Since this is a Jupyter Notebook, we can set environment variables cross-platform (works on Windows, Linux, and macOS) using Jupyter magic commands (`%env`) or Python's `os.environ` module.

## Method 1: Subprocess Command Line Execution (Recommended)
This method executes the script as a separate Python process using Jupyter's shell escape commands. It uses `%env` to set the GPU ID, making it cross-platform.

In [ ]:
# Set the CUDA visible devices (GPU index to use)
%env CUDA_VISIBLE_DEVICES=5

# Run the testing script with the parameters from test.sh
!python main_test.py --save_results --test_only --save test_1117_ms_re_v8_test --data_range 1-800/1-200 --save Full_results

## Method 2: Inline Python Execution
Alternatively, you can run the code directly within this Notebook process by overriding `sys.argv`. This approach shows the output directly inline and doesn't spawn a new subprocess.

In [ ]:
import os
import sys
import torch

# Set GPU device programmatically
os.environ['CUDA_VISIBLE_DEVICES'] = '5'

# Define arguments exactly as in the shell command
sys.argv = [
    'main_test.py',
    '--save_results',
    '--test_only',
    '--save', 'test_1117_ms_re_v8_test',
    '--data_range', '1-800/1-200',
    '--save', 'Full_results'
]

# Import and execute the main_test module
# Note: To re-run after changing arguments, restart the notebook kernel.
import utility
import data
import model
import loss
from option import args
from trainer import Trainer

# Initialize seed and checkpoint
torch.manual_seed(args.seed)
checkpoint = utility.checkpoint(args)

if checkpoint.ok:
    loader = data.Data(args)

    my_model = model.Model(args, checkpoint)
    # Load pretrained stage 1 weights
    my_model.model.load_state_dict(torch.load('pretrain/model_s1.pt'))

    args.n_colors = 3

    loss_fn = loss.Loss(args, checkpoint) if not args.test_only else None
    t = Trainer(args, loader, my_model, loss_fn, checkpoint)
    while not t.terminate():
        t.train()
        t.test()

    checkpoint.done()